## GPT n20 v01 gpt b10 run01 analysis

### requires python >= 3.10

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [3]:
DATA_FILE = "../../results/n20_examples_large_v01/gpt_v01/gpt_b10_run01.csv"

## Vastustega df

In [4]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")

### statistikat

In [23]:
counts2 = df1.groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,count
328,suutma,NaN,130
179,lugema,NaN,120
434,vastama,NaN,95
82,jääma,puudu,89
186,lõpetama,NaN,84
...,...,...,...
209,minema,järele,1
208,minema,alt,1
183,lähetama,NaN,1
67,istuma,koos,1


In [24]:
aggreg = df1.groupby(["verb", "verb_compound"], dropna=False).agg(
    yes_count=("classification", lambda x: np.sum(x == "yes")/len(x)*100),
        size = ("classification", lambda x :len(x))
).sort_values('yes_count', ascending=False)
aggreg

,,yes_count,size
verb,verb_compound,,
tõstatuma,NaN,100.0,1
istuma,koos,100.0,1
kallinema,NaN,100.0,2
tikkuma,ligi,100.0,3
söandama,NaN,100.0,1
...,...,...,...
lähetama,NaN,0.0,1
looma,välja,0.0,4
lülituma,välja,0.0,2


In [25]:
aggreg[aggreg["yes_count"]>= 80]

,,yes_count,size
verb,verb_compound,,
tõstatuma,NaN,100.000000,1
istuma,koos,100.000000,1
kallinema,NaN,100.000000,2
tikkuma,ligi,100.000000,3
söandama,NaN,100.000000,1
sõitma,tagasi,100.000000,1
hakkama,külge,100.000000,3
tulema,kaasa,100.000000,2
põruma,läbi,100.000000,2


In [26]:
aggreg[aggreg["yes_count"]<= 30]

,,yes_count,size
verb,verb_compound,,
saama,sisse,30.0,10
kolima,NaN,30.0,10
vaevama,NaN,30.0,10
kaduma,ära,30.0,20
saama,juurde,30.0,10
...,...,...,...
lähetama,NaN,0.0,1
looma,välja,0.0,4
lülituma,välja,0.0,2


In [19]:
aggreg = df1.groupby(["verb", "verb_compound", "morph_case"], dropna=False).agg(
    yes_count=("classification", lambda x: np.sum(x == "yes")/len(x)*100),
        size = ("classification", lambda x :len(x))
).sort_values('yes_count', ascending=False)
aggreg

,,,yes_count,size
verb,verb_compound,morph_case,,
astuma,ligi,all,100.0,2
dirigeerima,NaN,ad,100.0,3
möönma,NaN,ad,100.0,1
pilgutama,NaN,all,100.0,3
panema,välja,ad,100.0,7
...,...,...,...,...
vaatama,vastu,all,0.0,7
vajuma,lahti,ad,0.0,1
aktiviseeruma,NaN,ad,0.0,5


In [20]:
aggreg = df1.groupby(["verb", "verb_compound", "morph_case"], dropna=False).agg(
   no_count=("classification", lambda x: np.sum(x == "no")/len(x)*100),
        size = ("classification", lambda x :len(x))
).sort_values('no_count', ascending=False)
aggreg

no_count  size
verb          verb_compound morph_case                
ütlema        lahti         ad             100.0     2
              ära           ad             100.0     3
aktiviseeruma NaN           ad             100.0     5
vajuma        lahti         ad             100.0     1
vaatama       vastu         all            100.0     7
...                                          ...   ...
möönma        NaN           ad               0.0     1
hakkama       külge         all              0.0     3
dirigeerima   NaN           ad               0.0     3
tõstatuma     NaN           ad               0.0     1
astuma        ligi          all              0.0     2

[532 rows x 2 columns]

In [21]:
aggreg[aggreg["no_count"]>= 80]

no_count  size
verb          verb_compound morph_case                
ütlema        lahti         ad             100.0     2
              ära           ad             100.0     3
aktiviseeruma NaN           ad             100.0     5
vajuma        lahti         ad             100.0     1
vaatama       vastu         all            100.0     7
...                                          ...   ...
armuma        NaN           el              80.0     5
noogutama     NaN           all             80.0    10
kaasama       NaN           ad              80.0     5
vaagima       NaN           ad              80.0     5
vihjama       NaN           ad              80.0     5

[105 rows x 2 columns]

## ekilex=location

In [27]:
len(df1.loc[(df1["ekilex_tag"] == "location") ])

787

In [28]:
df1.loc[(df1["ekilex_tag"] == "location") & (df1["classification"] == "no")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
197,9909311,15895777,1,nimetama,NaN,el,pult,Puldist,"Puldist nimetas ta vaid seda , et kolmikliidu valitsusel on tulnud teha ebapopulaarseid otsuseid , sealhulgas koondada politseinikke .",NaN,location,NaN,no,NaN
409,3882671,6253266,13,soovitama,NaN,ad,NASA,NASA-l,"Astronaut ütles , et ta toetaks Kuule naasmist uurimiseesmärkidel , kuid soovitas NASA-l tõsiselt järele mõelda , kas sinna baasi loomine selleks , et sealt edasi Marsile minna , on mõttekas .",NaN,location,ORG,no,NaN
535,7668285,12295509,2,tunduma,NaN,ad,lõpumeeter,lõpumeetritel,"Ehkki lõpumeetritel tundus , et USA uus vabaujumise lootus Gary Hall jõudis venelasega ühele joonele , oskas Popov otsustava käelöögi hiilgavalt ajastada ning võita olümpia kroolisprindi finaali kohta selge vahe , 0,13 sekundiga .",NaN,location,NaN,no,NaN
938,8640736,13863622,6,võtma,vastu,abl,riiulifirma,riiulifirmadelt,Pealegi - miks toll võtab riiulifirmadelt piiril vastu eeldeklaratsioonid ?,NaN,location,NaN,no,NaN
1129,10436350,16737180,4,jääma,puudu,ad,allianss,alliansil,"Ainus , millest alliansil seekord puudu jäi , oli piisav rahvusvaheline toetus väljaspool NATOt .",NaN,location,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9233,1696299,2699975,3,võtma,NaN,ad,erafirma,erafirmal,KOHTUMARATON : erafirmal võttis kuus aastat sundimaks riiki võlga tunnistama,NaN,location,NaN,no,NaN
9247,16776755,25816087,1,soovitama,NaN,ad,Koigi,Koigil,"Koigil , kel on huvi asja vastu ja kes sooviksid suviti tootada lastelaagrites , soovitan minuga yhendust votta .",NaN,location,LOC,no,NaN
9388,7472181,11994213,4,aitama,NaN,ad,Mary,Maryl,"Proua Drusse aitab Maryl lõplikult surra ning see teeb Kuningriigist koha , kus on võimalik lootus ja jumaliku eksistentsi maine ümbersünd .",NaN,location,NaN,no,NaN
9597,4675656,7512628,7,jääma,vahele,all,keskkriminaalpolitsei,keskkriminaalpolitseile,Raoul Pajuviidik jäi ülisuure koguse narkootikumidega keskkriminaalpolitseile vahele 25. jaanuaril .,NaN,location,NaN,no,NaN


## vastus on koht= "yes" aga põhjenduses ütleb, et ei ole

In [30]:
yes = df1.loc[(df1["classification"] == "yes")]
conflict = yes.loc[(yes["explanation"].str.contains("not an adverbial")) | (yes["explanation"].str.contains("not a location")) | (yes["explanation"].str.contains("is not classified as adverbial of place"))]

In [31]:
conflict

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
32,12746239,20388907,3,suhtuma,NaN,adit,eliit,eliiti,Kõik suhtuvad eliiti põlgusega .,NaN,alive,NaN,yes,"The phrase 'eliiti' refers to a group rather than a physical location, so it is not an adverbial of place."
179,6939092,11151132,7,tõlkima,NaN,all,saadik,saadikutele,Eesti euroliitu viinud Hendrik Hololei tõlgib saadikutele peaminister Partsi kõnet .,NaN,NaN,NaN,yes,"The phrase 'saadikutele' (to the delegates) indicates the recipients of an action and does not specify a physical location, so it is not an adverbial of place."
194,10814828,17321127,11,kõlama,NaN,all,pankrotiohver,pankrotiohvritele,"Kuusel vedas , et tulema pääses : lausa mõnitusena kõlab pankrotiohvritele teade , et vahepeal kombinaadilt saadud pisukegi palgasumma võidakse kohtu kaudu tagasi nõuda .",NaN,NaN,NaN,yes,"The phrase 'pankrotiohvritele' refers to a group of people (bankruptcy victims), not a location, so it is not classified as an adverbial of place."
208,6270282,10072883,9,töötama,NaN,el,liikluskorraldus,liikluskorraldusest,"Pealinnas töötab aga kõik selle vastu , alates liikluskorraldusest kuni kaasliiklejateni .",NaN,NaN,NaN,yes,"The phrase 'liikluskorraldusest' indicates an abstract concept related to traffic management, not a specific place, hence not an adverbial of place."
213,8705140,13965151,20,meenutama,NaN,all,Laura,Laurale,"Näiteks see vasakus reaservas istuv heledapäine apollolokkidega noormees , kellele ta valge tärgeldatud krae ilmselgelt ebamugavust valmistas , meenutas Laurale vägisi üht innukat maapoissi , kes arvas vist , et linnas leiab ta eest hoopis kõrgema inimliigi esindajad , ja kes oleks armunud ilmselt esimesesse ettejuhtuvasse vähegi sobilikku naisterahvasse — puhtjuhuslikult oli Laura see , kellele ta oma naiivse tuhina pühendas .",NaN,location,PER,yes,"The phrase 'Laurale' indicates direction or target rather than a physical location; it does not describe a place; therefore, it is not an adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9799,9320711,14971746,18,püüdma,NaN,all,katsumus,katsumusele,"Esmaspäeval Prantsusmaa jalgpallimeistri RC Lensi prooviajale siirduv FC Flora leedulane Tomas Razhanauskas kinnitas , et püüab tähtsale katsumusele eriti mitte mõelda : “ Lähen kohale ja teen , mis oskan . ”",NaN,event,NaN,yes,"The phrase 'katsumusele' refers to a metaphorical target or goal rather than a physical place, so it is not an adverbial of place."
9887,7264666,11681489,1,tundma,NaN,el,hingesoojus,Hingesoojusest,"Hingesoojusest tundsid lapsed tegelikult kõige rohkem puudust , "" sõnas Malle Kruusma .",NaN,NaN,NaN,yes,"The word 'Hingesoojusest' does not describe a physical place but rather an abstract concept, so it is not an adverbial of place."
9974,4320683,6950745,19,pühendama,NaN,all,firma,firmale,"Tema sõnul on ettevõtte algstaadium firma tegevuse edasise arengu seisukohalt väga oluline , ning just siis pühendab juht firmale kogu oma energia .",NaN,NaN,NaN,yes,"The phrase 'firmale' indicates the entity being dedicated energy, and does not specify a physical location, hence it is not an adverbial of place."
9975,6983956,11219813,24,laulma,NaN,all,kirjanik,kirjanikule,"“ Kui saatus viib ka kodust kaugele , toob ükskord tagasi meid tee , ” laulis naisansambel Klassik Maie Kala klaverihelide saatel auväärsele kirjanikule juubeliõnnitluseks ning noorpõlveradadele naasmise puhuks .",NaN,alive,NaN,yes,"The phrase 'kirjanikule' specifies a recipient rather than a location, so it is not an adverbial of place."


## "no" vastuste seas ei tundu selliseid konflikte olevat

In [34]:
no = df1.loc[(df1["classification"] == "no")]
conflict2 = no.loc[(no["explanation"].str.contains("is an adverbial of place")) | (no["explanation"].str.contains("is a location")) | (no["explanation"].str.contains("is classified as adverbial of place"))]

In [35]:
conflict2

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation


## n20 yes vastused

In [7]:
yes = df1.loc[(df1["classification"] == "yes")]
yes

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
11,13567621,21694794,10,nimetama,NaN,el,esinelik,esinelikust,Hooaja jooksul on Klandorf mitu korda nimetanud A . Le Coqi esinelikust meelepäraseimaks vastaseks .,NaN,NaN,NaN,yes,"The phrase 'esinelikust' refers to a specific place or group, making it an adverbial of place."
12,12825358,20516338,3,saama,üle,el,5.05,5.05st,"Mõlemad said 5.05st üle , kolm ebaõnnestunud katset tegi Šebrle 5.15-l ja sellest kõrgusest loobunud Nool 5.25-l.",NaN,NaN,NaN,yes,"The phrase '5.05st' indicates a location related to a height, classifying it as an adverbial of place."
15,367679,568736,26,püüdma,NaN,all,kõver,kõverale,"Semper arutleb , et Norra mägimaastikele iseloomulik kõverjoon on omane Knut Hamsuni tegelastele ; Ibseni traagika arvab ta peituvat seigas , et see püüdis Norra kõverale joonele peale suruda õgujoone külmjulma loogikat .",NaN,NaN,NaN,yes,"The phrase 'kõverale' specifies a location in a symbolic or illustrative manner, thus it qualifies as an adverbial of place."
18,6186079,9932912,19,korraldama,NaN,all,kontor,kontorile,"Regionaalprokurör Juri Ketov lausus , et relvastatud rühmitus korraldas rünnaku siseministeeriumile ja regionaalsele föderaalse julgeolekubüroo ( FSB ) kontorile .",NaN,location,NaN,yes,"The phrase 'kontorile' specifies a physical place, making it an adverbial of place."
22,6313845,10144762,11,pajatama,NaN,el,mäss,mässust,"Rääkimata romaanist "" Waverly "" , mis pajatab 1745. aasta mässust , mille käigus šotlased üritasid oma meest Inglismaa troonile tagasi panna .",NaN,NaN,NaN,yes,"The phrase 'mässust' refers to the uprising and functions to specify the location of an event in relation to the context, which is why it is classified as adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9991,1784481,2841912,5,näitama,NaN,ad,kodukas,kodukal,"Peep Rada näitab oma kodukal ( ) , milliseid imelisi asju on näha Eesti veealuses maailmas .",NaN,NaN,NaN,yes,"The phrase 'kodukal' describes a specific place ('on the homepage'), making it adverbial of place."
9992,9140347,14694341,18,armuma,NaN,ill,telediktor,telediktorisse,Kolman ( Lembit Ulfsak ) on oma naisest ( Maria Klenskaja ) tüdinud ja armub hoopis noorde telediktorisse ( Kristel Sarnet ) .,NaN,alive,NaN,yes,"The phrase 'telediktorisse' describes the place or entity into which someone falls in love, thus classified as adverbial of place."
9993,1716920,2733114,4,nimetama,NaN,ad,mina,meil,"Ma ei nimetaks meil toimunut koondamiseks , vaid töö efektiivsemaks muutmiseks .",NaN,NaN,NaN,yes,"The phrase 'meil' indicates a location ('with us' or 'at our place'), making it adverbial of place."
9996,4898982,7865678,13,jääma,kõrvale,el,seanss,seanssidest,"Kui aga tegemist on abikaasade omavaheliste probleemidega , jäävad lapsed reeglina neist seanssidest kõrvale .",NaN,NaN,NaN,yes,"The phrase 'seanssidest' describes a location or context from which someone is excluded, thus classified as adverbial of place."


In [30]:
## kui paljudes on numbrid: 14
df1.loc[(df1["classification"] == "yes") & (df1["form"].str.contains(r"\d", na=False))]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
12,12825358,20516338,3,saama,üle,el,5.05,5.05st,"Mõlemad said 5.05st üle , kolm ebaõnnestunud katset tegi Šebrle 5.15-l ja sellest kõrgusest loobunud Nool 5.25-l.",NaN,NaN,NaN,yes,"The phrase '5.05st' indicates a location related to a height, classifying it as an adverbial of place."
758,4704426,7558898,5,läbima,NaN,ad,C19,C19-l,Nick Heidfeld läbis vastavalt C19-l 64 ringi .,NaN,NaN,NaN,yes,"The word 'C19-l' indicates a specific location (on the car), which makes it an adverbial of place."
4420,11391921,18254895,29,meenutama,NaN,all,TVN24,TVN24-le,""" Ajal , mil ma koomasse langesin , oli poodides vaid sinep ja äädikas , liha sai talongidega ja igal pool olid hiiglaslikud bensiinijärjekorrad , "" meenutas Grzebski TVN24-le kommunistliku süsteemi kokkuvarisemise aega .",NaN,NaN,NaN,yes,"It indicates the specific place where the recollection was shared, making it an adverbial of place."
4733,443342,693588,9,juhtima,NaN,ad,470.,470-l,Neusiedli järve ( Austria ) purjeregatil juhivad naiste 470-l pärast 8 sõitu Maria Veessaar/Maiki Saaring .,NaN,NaN,NaN,yes,"The phrase '470-l' refers to a specific location or area in which the activity takes place, thus making it an adverbial of place."
5069,6440037,10353921,5,sõltuma,NaN,el,CO2,CO2-st,""" Põlevkivielektri tulevik sõltub CO2-st , "" tunnistas Eesti Energia arendusjuht Indrek Aarna : "" Juhul kui CO2 kvoote kärbitakse , siis järgmistel renoveeritavatel põlevkivikateldel ei ole enam suurt tulevikku . """,NaN,NaN,NaN,yes,The phrase 'CO2-st' was classified as an adverbial of place ('yes') because it specifies the originating context or source regarding CO2 in a way that implies a metaphorical spatial relationship.
5392,19679024,29232070,6,arvama,NaN,el,C4U,C4U -st,Johen: mida sa arvad C4U -st,NaN,NaN,NaN,yes,"The phrase 'C4U -st' implies origin or source, which can signify a location, qualifying it as an adverbial of place."
5744,17660897,26998555,36,piisama,NaN,el,100g,100g'st,"Sega kokku köharohi , penitsilliin , piiritus ja tavaline vesi suhtes 3:4:2:1 , kokku peab olema vedelikku 1l , või kui on väiksem neet/rõngas siis piisaks ka 100g'st .",NaN,NaN,NaN,yes,The phrase '100g'st' is classified as adverbial of place because it indicates a quantitative base that indirectly relates to the context of measurement.
6525,2065875,3296016,19,tegema,NaN,ad,3,3L,"Soome sõnul andis ta Kanada kalandusjuhile üle väga detailse materjali , mida iga Eesti laev eraldi on tsoonis 3L teinud .",NaN,NaN,NaN,yes,"The phrase '3L' refers to a specific zone, describing a location connected with the action, which makes it adverbial of place."
6899,6616615,10647898,7,saama,üle,el,2.23,2.23-st,"Esimesed katsed olid küll kurjakuulutavad , 2.23-st sain üle alles kolmandal katsel .",NaN,NaN,NaN,yes,"The phrase '2.23-st' indicates the point or location (in a figurative sense) from which something was surpassed, so it is classified as an adverbial of place."
8298,4971855,7983572,4,läbima,NaN,ad,F1-auto,F1-autol,Siiani on Räikkönen F1-autol läbinud 4500 km .,NaN,NaN,NaN,yes,"The phrase 'F1-autol' specifies the location where the action happened, justifying its classification as an adverbial of place ('yes')."


In [8]:
yes.groupby(['morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,morph_case,count
1,ad,2249
4,el,938
3,all,922
0,abl,140
5,ill,58
6,in,24
2,adit,22


In [9]:
yes.loc[(yes["morph_case"] == "adit")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
32,12746239,20388907,3,suhtuma,NaN,adit,eliit,eliiti,Kõik suhtuvad eliiti põlgusega .,NaN,alive,NaN,yes,"The phrase 'eliiti' refers to a group rather than a physical location, so it is not an adverbial of place."
422,12068663,19318667,13,suhtuma,NaN,adit,hukkamine,hukkamisse,"Aga ajalugu , kultuuri ja rahvusvahelist õigust mõistvad inimesed ei suhtu epohhiloovasse hukkamisse sugugi nii must-valgelt .",NaN,NaN,NaN,yes,"The phrase 'hukkamisse' describes a location or direction (to condemnation), thus it is an adverbial of place."
700,7152563,11504788,17,suhtuma,NaN,adit,ilukirjandus,ilukirjandusse,"Et Larry teadis ja oli lugenud mingit teisejärgulist ingliskeelset teost – tema , kes ta suhtus ilukirjandusse alati teatud iroonia ja üleolekuga .",NaN,NaN,NaN,yes,"Classified as 'yes' because 'ilukirjandusse' indicates a direction or location related to literature, functioning as an adverbial of place."
954,2534772,4066634,7,suhtuma,NaN,adit,ettevalmistus,ettevalmistusse,Kaks aastat tagasi alustatud kõrghariduse reformi ettevalmistusse suhtusid osapooled enneolematu entusiasmiga .,NaN,NaN,NaN,yes,"It was classified as 'yes' because 'ettevalmistusse' indicates the direction or location into which the preparation was made, functioning as an adverbial of place."
1263,7834913,12554635,9,suhtuma,NaN,adit,koostööpakkumine,koostööpakkumisse,"Ka Leps märgib , et Savisaar suhtus tema koostööpakkumisse jahedalt .",NaN,NaN,NaN,yes,"The phrase ""koostööpakkumisse"" indicates movement or direction into something, classifying it as an adverbial of place."
1453,3672154,5913249,3,suhtuma,NaN,adit,Maapank,Maapanka,""" Suhtume Maapanka kui eraldi seisvasse üksusesse ja pole kellegagi läbirääkimisi pidanud .",NaN,NaN,ORG,yes,"The phrase 'Maapanka' specifies a directional or locational aspect by referring to a particular place, classifying it as adverbial of place."
2101,4358027,7010589,7,suhtuma,NaN,adit,esitus,esitusse,Siiski suhtuvad lätlased oma hokikoondise äsjasesse esitusse pigem soosivalt .,NaN,event,NaN,yes,"The phrase 'esitusse' ('to the presentation') indicates a direction related to place, hence it is adverbial of place."
2139,355518,549532,3,suhtuma,NaN,adit,ajakirjandus,ajakirjandusse,"Olen siiani ajakirjandusse neutraalselt suhtunud , sest see ei olnud mind isiklikult kunagi puudutanud .",NaN,NaN,NaN,yes,"The phrase 'ajakirjandusse' denotes a direction or target (into journalism), making it an adverbial of place."
2803,8256099,13245664,4,suhtuma,NaN,adit,Jeruusalemm,Jeruusalemma,"Praegused arhitektid suhtuvad Jeruusalemma rohkem kui maastikku , linna arendamise strateegiaks on konservatiivne urbanism .",NaN,location,LOC,yes,"Classified as 'yes' because 'Jeruusalemma' specifies the location being discussed, indicating place."
3440,107884,181516,8,suhtuma,NaN,adit,avastus,avastusse,"Polnud niisiis ime , et üldsus suhtus avastusse umbusuga .",NaN,NaN,NaN,yes,"The phrase 'avastusse' specifies the place where the public's distrust is directed towards, thus it is adverbial of place."


In [10]:
yes.loc[(yes["morph_case"] == "ad")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
24,14061675,22389273,11,tulema,kohale,ad,Mühlenbeck,Mühlenbeckil,"SV Mühlenbecki ja Birkenwerder BC mängu algul selgus , et Mühlenbeckil tuli kohale kuus meest .",NaN,NaN,LOC,yes,"The phrase 'Mühlenbeckil' refers to the location where an event happened (e.g., the state of the team at a particular site), making it adverbial of place."
25,8247308,13232622,9,kuulama,NaN,ad,õu,õuel,"Meie ei ütle midagi , meie kuulame lummemattunud õuel hämmeldunult märtsikuist punarinna laulu , mis ei saa küll kuidagimoodi olla alamlaul , vaid ikka ülemlaul , sest Juhan Liiv ju kirjutas , et",NaN,NaN,NaN,yes,"The phrase 'õuel' specifies the location where something is being heard, showing its function as an adverbial of place."
29,685016,1089414,12,võistlema,NaN,ad,etapp,etapil,Peaaegu kolm kuud tervist taastanud Michael Schumacher võistleb F1-e kahel viimasel etapil : Malaisias 17. oktoobril ning Jaapani Suzukas 31. oktoobril .,NaN,NaN,NaN,yes,"The phrase 'etapil' points to specific stages of a race, describing the location of the events, so it is an adverbial of place."
30,5053218,8109495,13,kasutama,NaN,ad,kaubamärk,kaubamärgil,"Omaniku valitud uus rentnik , osaühing Wostock Oil , kasutab sirpi-vasarat oma kaubamärgil .",NaN,NaN,NaN,yes,"The phrase 'kaubamärgil' refers to a location where the sirpi-vasarat symbol is used, making it an adverbial of place."
31,9105951,14641235,14,ärkama,NaN,ad,keskväljak,keskväljakul,"Eesti suurim , 7000-kohaline vabaõhu ooperiteater ärkab suvel üheks päevaks ellu Tõrva linna keskväljakul .",NaN,location,NaN,yes,"The phrase 'keskväljakul' describes a location where the ooperiteater comes to life, qualifying it as an adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9979,3357501,5401117,4,pruukima,NaN,ad,Levadia,Levadial,"Seetõttu ei pruugi Levadial skoori avamiseks liiga palju võimalusi tekkida ning taas võib otsustavaks saada pealinlaste keskväljale koondunud kahurvägi , kes vastaste väravat ka kaugemalt väga efektiivselt pommitada suudab .",NaN,NaN,LOC,yes,"The phrase 'Levadial' specifies the entity being discussed rather than a physical location, so it is not an adverbial of place."
9980,117383,197351,3,läbima,NaN,ad,maa,maal,Mullu läbis maal sellise “ kadalipu ” paar tuhat endist kolhoosnikku-sovhoosnikku .,NaN,NaN,NaN,yes,It is classified as an adverbial of place because 'maal' ('on the land/countryside') indicates a location.
9991,1784481,2841912,5,näitama,NaN,ad,kodukas,kodukal,"Peep Rada näitab oma kodukal ( ) , milliseid imelisi asju on näha Eesti veealuses maailmas .",NaN,NaN,NaN,yes,"The phrase 'kodukal' describes a specific place ('on the homepage'), making it adverbial of place."
9993,1716920,2733114,4,nimetama,NaN,ad,mina,meil,"Ma ei nimetaks meil toimunut koondamiseks , vaid töö efektiivsemaks muutmiseks .",NaN,NaN,NaN,yes,"The phrase 'meil' indicates a location ('with us' or 'at our place'), making it adverbial of place."


## n20 no vastused

In [11]:
no = df1.loc[(df1["classification"] == "no")]
no

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,7108634,11432411,7,tasuma,NaN,ad,kord,korral,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",NaN,NaN,NaN,no,"The phrase 'korral' refers to 'occasion' and does not indicate a location, hence it is not adverbial of place."
1,2601827,4175718,12,pakkuma,NaN,ad,juhatus,juhatusel,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",NaN,NaN,NaN,no,"The phrase 'juhatusel' refers to 'under the direction' and is describing leadership or guidance, not location, so it is not adverbial of place."
2,1677278,2670999,3,seletama,NaN,all,patsient,patsiendile,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",NaN,NaN,NaN,no,NaN
3,750912,1196732,14,virutama,NaN,all,reisija,reisijale,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",NaN,alive,NaN,no,NaN
4,4733683,7604499,14,lülitama,NaN,ad,poolaeg,poolajal,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,NaN,time,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,12758554,20409273,5,meenuma,NaN,all,Andrew,Andrew'le,Uue plaadi tulekuga meenub Andrew'le veel üks probleem .,NaN,NaN,PER,no,"The phrase 'Andrew'le' refers to a person receiving something, but it does not describe a location, so it is not adverbial of place."
9994,1715798,2731307,14,lõpetama,NaN,el,isa,isast,"Ma otsutasin , et kui lööb ette aasta 2000 , lõpetan ma oma isast rääkimise .",NaN,alive,NaN,no,NaN
9995,9799023,15722082,8,käsitlema,NaN,ad,päev,päevil,Film käsitleb sündmusi 1943. aastal Varssavis kannatusnädala päevil ja põhineb Jerzy Andrzejewski romaanil .,NaN,NaN,NaN,no,NaN
9997,12893712,20625029,4,saama,sisse,ad,esitamine,esitamisel,Riigikogulased saavad töötõendi esitamisel tasuta sisse .,NaN,NaN,NaN,no,NaN


In [12]:
no.groupby(['morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,morph_case,count
1,ad,3056
4,el,1300
3,all,1038
0,abl,104
6,in,80
2,adit,56
5,ill,13


In [13]:
no.loc[(no["morph_case"] == "ill")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
1005,11882961,19014665,4,armuma,NaN,ill,muster,mustritesse,Disainerid on armunud mustritesse .,NaN,NaN,NaN,no,NaN
1027,9018186,14499673,6,süstima,NaN,ill,õpilane,õpilastesse,"Kuigi lehe tegemise idee on õpilastesse enamasti süstinud õpetajad , teevad noored seda suure innuga .",NaN,alive,NaN,no,NaN
2907,3990163,6422749,17,armuma,NaN,ill,meditsiiniõde,meditsiiniõesse,"Sel ajal kui mitmed meeskolleegid talle külge proovivad lüüa , armub ta ise seriaalis kaasa tegevasse meditsiiniõesse Juliesse ( Jessica Lange ) .",NaN,alive,NaN,no,NaN
2945,9165169,14733329,3,sekkuma,NaN,ill,palve,palvesse,Tihti sekkub palvesse “ kerge ooperi ” ülemeelik vaim .,NaN,NaN,NaN,no,NaN
3338,3414249,5495283,15,süstima,NaN,ill,kes,kellesse,"Esimesi eksponaate hakkasid muuseumi tarvis alates 1993. aastast tallele panema Tallinna politseikooli kadetid , kellesse politsei ajalugu uuriv Mai Krikk süstis kogumisvaimustust .",NaN,NaN,NaN,no,NaN
4432,4065171,6543605,38,sekkuma,NaN,ill,laekumine,laekumistesse,"Linnavolinikest koosneva revisjonikomisjon 26. märtsi akt Valika kontrollimisest ütleb : "" Transpordiamet , selle asemel , et nõuda kogu laekuva parkimistasu ülekandmist linnaeelarvesse , kehtestas omavoliliselt oma käskkirjaga Valika kohustusi alusetult piirava maksegraafiku , millega sekkus linnaeelarve laekumistesse ja ületas oma pädevust . """,NaN,NaN,NaN,no,NaN
5797,10018978,16071762,5,armuma,NaN,ill,vanahärra,vanahärrasse,Üks vanaproua armus hiljuti vanahärrasse .,NaN,alive,NaN,no,NaN
6081,19056972,28756261,6,sekkuma,NaN,ill,erahuvi,erahuvidesse,"Kui riik sekkub siin eraisiku erahuvidesse ja keelab tal maad müüa sellele isikule , kellele ta tahab , paneb ta vanainimese jätkuvalt sellessesamasse olukorda , kus ta peab oma vanaduspäevad sama kesiselt mööda saatma kui seni , olles õnnelik , et ta on maaomanik .",NaN,NaN,NaN,no,NaN
6620,6382033,10257564,26,süstima,NaN,ill,kooma,koomasse,"Käesolevaks aastaks prognoosib Tallinna Kiirabi väljakutsete arvu üle 400 , kiirabi peaarsti Raul Adlase sõnul teevad nende seas ilma narkomaanid , kes on ennast korduvalt koomasse süstinud .",NaN,NaN,NaN,no,"The phrase 'koomasse' refers to a state or condition, not a place, so it is not an adverbial of place."
7490,12294833,19681594,16,sekkuma,NaN,ill,esikohaheitlus,esikohaheitlustesse,"Tartu Triitoni , Tallinna noortespordikeskuse , Saku ja Nõo sulgpalliklubi esindajate kõrval sekkusid kahes vanuserühmas esikohaheitlustesse ka külalised Lätimaalt .",NaN,NaN,NaN,no,"The phrase 'esikohaheitlustesse' indicates involvement in an activity rather than a specific location, so it is not an adverbial of place."


In [43]:
no.loc[(no["morph_case"] == "in")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
555,3300135,5305255,21,kahtlema,NaN,in,koostöötahe,koostöötahtes,"TALLINN , 21. aprill ( EPLO ) - Riigikogu Keskerakonna fraktsiooni aseesimees ja erakonna juhatuse liige Ain Seppik kahtleb siseopositsionääride koostöötahtes .",NaN,NaN,NaN,no,NaN
556,5723806,9180161,19,vaevlema,NaN,in,konkurents,konkurentsis,"Ei saa ju olla nii naiivne ja mõelda , et kasumile suunatud reisikorradusfirmad ja niigi kogu aeg kasvavas konkurentsis vaevlevad lennufirmad selle enda südameasjaks võtavad .",NaN,NaN,NaN,no,NaN
619,9407096,15105649,16,vaevlema,NaN,in,stress,stressis,"Siis saaks neile selgemaks , miks on iive nii väike , miks vaevleb rahvas pidevas stressis , miks sagenevad suitsiidid ja kuritegevus .",NaN,NaN,NaN,no,NaN
640,6758994,10873757,6,süüdistama,NaN,in,enamik,enamikus,"USA armee süüdistab neis surmades enamikus endist Iraagi diktaatorit Saddam Husseini toetavaid isikuid , kuid suureneb ka välisterroristide osalus neis aktides .",NaN,NaN,NaN,no,"The phrase 'enamikus' does not describe a location but rather refers to a portion or majority, so it is not an adverbial of place ('no')."
740,3130432,5027269,11,süüdistama,NaN,in,evimatus,evimatuses,"Araabiakeelse ajalehe al-Sharq al-Awsat teatel süüdistab Liibüa Iisraeli poliitilise eetika evimatuses rahvusvahelistes suhetes , teatas Ha'aretz .",NaN,NaN,NaN,no,"The phrase 'evimatuses' refers to a state or condition rather than a location, so it is not an adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8791,3280575,5272745,5,vaevlema,NaN,in,sisepinge,sisepingetes,"Viimased kuu aega suurtes sisepingetes vaevelnud Keskerakonna toetus kukkus 13 protsendile , mis on Eesti suurima erakonna jaoks viimase kolme aasta nõrgim näitaja .",NaN,NaN,NaN,no,NaN
8831,5977421,9592942,6,süüdistama,NaN,in,isekus,isekuses,"Euroopa Liidu riigid süüdistasid Suurbritanniat isekuses , sest Londoni esmaspäeval esitletud eelarve-ettepanek teenivat üksnes brittide endi huve .",NaN,NaN,NaN,no,NaN
8889,9088541,14613797,3,vaevlema,NaN,in,ajahäda,ajahädas,"“ Aga ajahädas vaevlevad kõik tiimid , mitmed ei jõudnudki tänasesse vabatrenni .",NaN,NaN,NaN,no,NaN
9278,18375889,27858036,6,kahtlema,NaN,in,loodusseadus,loodusseadustes,"Aga ma pole kunagi kahelnudki loodusseadustes ja teaduse võimes neid seadusi avastada , üha sügavamalt mõista ja kasutada , 33338 !",NaN,NaN,NaN,no,NaN


In [45]:
no[no["morph_case"]=="in"].groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,count
0,kahtlema,NaN,33
1,süüdistama,NaN,29
2,vaevlema,NaN,18


In [46]:
no[no["morph_case"]=="ad"].groupby(['verb','verb_compound'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)


,verb,verb_compound,count
51,jääma,puudu,40
224,tasuma,NaN,37
182,põhinema,NaN,37
241,tulema,NaN,34
121,lubama,NaN,34
...,...,...,...
216,sõitma,ära,1
253,tõstma,esile,1
234,tsiteerima,NaN,1
63,kaebama,NaN,1


In [48]:
no[(no["morph_case"]=="ad") & (no["verb"]=="jääma") & (no["verb_compound"]=="puudu")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
116,2770595,4443043,25,jääma,puudu,ad,eestlane,eestlastel,"Kultuuriteoreetik Anzori Barkalaja usub , et praegune seosetu rahvussümbolitega vehkimine näitab vaid seda , et üleilmastumise ja isikliku kasu tagaajamise ajal jääb üha enamatel eestlastel millestki olulisest puudu .",NaN,alive,NaN,no,NaN
139,5939167,9534052,10,jääma,puudu,ad,Stanišev,Staniševil,"Kui aga parlament pidi kinnitama valitsuse koosseisu , jäi Staniševil üks hääl puudu .",NaN,NaN,LOC,no,NaN
201,2134739,3409765,5,jääma,puudu,ad,Netšeporuki,Netšeporukil,"Oma Eesti rekordist jäi Netšeporukil puudu 21 punkti , siiski on Talence'i võidutulemus maailma hooaja edetabelis kümnes .",NaN,NaN,LOC,no,NaN
670,9806624,15734310,4,jääma,puudu,ad,Ustritsk,Ustritskil,"88. minutil jäi Ustritskil vähe puudu , et peaga võiduvärav lüüa .",NaN,NaN,LOC,no,NaN
705,3080216,4944265,3,jääma,puudu,ad,meeskond,meeskonnal,""" Pärnu meeskonnal jääb pisut puudu rünnakuvõimsusest , kuigi olen kindel , et nädal tagasi Moskvas peetud kohtumises kogu potentsiaal ei avanenud .",NaN,alive,NaN,no,NaN
1129,10436350,16737180,4,jääma,puudu,ad,allianss,alliansil,"Ainus , millest alliansil seekord puudu jäi , oli piisav rahvusvaheline toetus väljaspool NATOt .",NaN,location,NaN,no,NaN
1297,15633963,24392695,14,jääma,puudu,ad,linnainimene,linnainimestel,"Tillukesest Vajangust pärit maatüdrukuna hindab ta inimeste juures siirust ja leiab , et linnainimestel jääb sellest omadusest veidi puudu .",NaN,alive,NaN,no,NaN
1762,498554,784015,5,jääma,puudu,ad,kiirendamine,kiirendamisel,"Maanteel jääb suuremate käikudega kiirendamisel veidi jõust puudu , kuid sellise litraazhi puhul ei saagi enamat soovida .",NaN,NaN,NaN,no,NaN
1866,14726412,23200254,8,jääma,puudu,ad,Nikolai,Nikolail,"” Viimane matš ei olnud lootusetu , Nikolail jäi puudu kogemustest , ” hindas Joffe debütandi etteastet .",NaN,NaN,PER,no,NaN
2151,11296131,18103231,14,jääma,puudu,ad,juhtimine,juhtimisel,"Selline rõhuasetus näitab , et poliitilistest teadmistest ja tulevikku suunatud mõtlemisest jääb rahandusministeeriumi juhtimisel oluliselt puudu .",NaN,NaN,NaN,no,"It was not classified as adverbial of place because 'juhtimisel' refers to the action of management, not a physical location."


## numbrid

In [151]:
num_idx = (no["form"].str.contains(r"\d", na=False))
no.loc[num_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
231,16315936,25261416,5,kirjutama,NaN,ad,n2da,n2dal,Kirjutan siia peaaegu iga n2dal yhte ja sama jutu mis l2heb kogu aeg edasi ja see lugu on zombidest ...,NaN,NaN,NaN,no,NaN
394,12500346,20008752,8,hääletama,NaN,el,95,95st,Enam kui seitse tundi väldanud koosolekul hääletas 95st kohal olnud TKLi liikmest 50 Liivati ja 55 Nurga tagandamise poolt .,NaN,NaN,NaN,no,NaN
635,17561216,26917928,15,minema,NaN,ad,90.,90ndail,"Nõukogude ajal toimetas seal Haapsalu tarbijate ühistu , Vormsi tarbijate ühistule läks pood üle 90ndail .",NaN,NaN,NaN,no,NaN
1158,12659626,20255912,1,tulema,NaN,ad,1975.,1975ndal,1975ndal tuli MM-tiitel .,NaN,NaN,NaN,no,NaN
1289,14417303,22834097,13,saama,NaN,ad,16.,16ndal,"Tunned , et trenn on praegu rohkem töö kui lõbu , kuid 16ndal saad uut energiat sellest , et satud millegagi tähelepanu keskpunkti .",NaN,NaN,NaN,no,NaN
1648,18532170,28058688,31,võtma,NaN,ad,10.,10l,"See , et ta linnasõidul 11l/100-le võtab on tegelikult normaalne - ja kui pika maa peal sellise viimase aja lumepudru sees ( olgu see lumekiht kasvõi 5mm ) 10l/100-le võtab , siis pole ka veel hullu lahti .",NaN,NaN,NaN,no,NaN
1860,8736255,14023001,11,piisama,NaN,el,33,33st,"Kokku teeb see vähemalt 35 vastuhäält , ehkki piisab juba 33st .",NaN,NaN,NaN,no,"The phrase '33st' indicates a quantity and does not describe a location, so it is not an adverbial of place."
2268,1279830,2033951,7,esinema,NaN,ad,3%,3%-l,"On leitud , et roojamishäiret esineb 3%-l lastest .",NaN,NaN,NaN,no,NaN
2807,5542339,8887042,13,töötama,NaN,el,600.,600-st,Eile hommikul ei töötanud elektrikatkestuste tõttu ka ligi 80 Eesti Mobiiltelefoni tugijaama 600-st ning 10-15 protsenti Radiolinja ja pisut alla 10 protsendi Tele2 tugijaamadest .,NaN,NaN,NaN,no,NaN
2825,11471325,18378431,3,piisama,NaN,el,"49,02","49 , 02-st","Võiduks piisas 49 , 02-st .",NaN,NaN,NaN,no,NaN


## elus 

In [64]:
# -lane lõpuline
lane_idx = no["lemma"].str.endswith("lane")
no[lane_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
116,2770595,4443043,25,jääma,puudu,ad,eestlane,eestlastel,"Kultuuriteoreetik Anzori Barkalaja usub , et praegune seosetu rahvussümbolitega vehkimine näitab vaid seda , et üleilmastumise ja isikliku kasu tagaajamise ajal jääb üha enamatel eestlastel millestki olulisest puudu .",NaN,alive,NaN,no,NaN
255,7404632,11886864,17,nõudma,NaN,ad,kreeklane,kreeklastel,"Mina jooksin pardale , käskisin hiivata ankruköie ning kui laeva põhja alt kostis kopsimist , nõudsin kreeklastel ettevaatlikult põhja auk saagida .",NaN,alive,NaN,no,NaN
709,13839283,22066709,12,käskima,NaN,ad,sugulane,sugulastel,"Mõtlesin isegi , et äkki Juri polegi surnud , vaid käskis sugulastel seda lihtsalt minu Venemaale meelitamiseks öelda . ”",NaN,NaN,NaN,no,NaN
741,5197195,8337470,4,ebaõnnestuma,NaN,ad,sakslane,sakslasel,"MK teisel , sakslasel Martin Schmittil ebaõnnestus võistlus Sapporos .",NaN,alive,NaN,no,NaN
791,194329,320429,5,valmistama,NaN,all,arvutiteadlane,arvutiteadlastele,"Viiendaks , ja mis arvutiteadlastele eriti meelehärmi valmistab , on meid ümbritseva maailma pidevus , mistõttu on raske kõiki võimalikke sisendeid ja väljundeid korralikult ära nummerdada .",NaN,alive,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9633,3928827,6327499,17,määrama,NaN,all,kaitseväelane,kaitseväelasele,"TALLINN , 25. märts ( EPLO ) - Kaitseväe juhataja viitseadmiral Tarmo Kõuts määras distsiplinaarkaristuse seitsmele kaitseväelasele , kes tarvitasid Sharjah' emiraadi lennujaamas alkoholi , kirjutab Postimees .",NaN,alive,NaN,no,NaN
9668,4973468,7985850,16,keelduma,NaN,el,erusõjaväelane,erusõjaväelasest,""" Meil pole isegi lapsendusõigust , muust rääkimata , "" ohkab Evi Suurkivi-Bogale , kelle erusõjaväelasest etioopia mehele keeldub Eesti riik alalist elamisluba andmast .",NaN,NaN,NaN,no,NaN
9826,4648052,7469792,2,soovitama,NaN,all,kooliõpilane,kooliõpilastele,Kuid kooliõpilastele ja teistele suurematele turismigruppidele soovitab Kork külastada Akste asemel Kiidjärve metskonda jäävaid teisi sipelgaasurkondi .,NaN,alive,NaN,no,NaN
9837,19088311,28801164,20,tundma,NaN,el,fraktsioonikaaslane,fraktsioonikaaslastest,"Ma usun , et kui ka sina oleksid kaks vahetust volikogus töötanud , siis tunneksid sa seda valdkonda oma fraktsioonikaaslastest kõige paremini .",NaN,NaN,NaN,no,NaN


In [83]:
# inimesed
live_idx1 = (no["lemma"].str.endswith("arst")) | (no["lemma"]=="ema") | (no["lemma"]=="isa")
live_idx2 = (no["lemma"]=="vanaema") | (no["lemma"]=="vanaisa") | (no["lemma"]=="õde") | (no["lemma"]=="vend")
live_idx3 = (no["lemma"]=="tädi") | (no["lemma"]=="onu") |  (no["lemma"]=="mees") | (no["lemma"]=="naine")

no[live_idx1 | live_idx2 | live_idx3]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
91,421670,657279,7,võtma,ära,abl,mees,mehelt,Maardu politseinikud võtsid esmaspäeval Maardus kahelt mehelt ära neli kotikest arvatavasti narkootilist ainet .,NaN,NaN,NaN,no,NaN
100,355575,549627,12,mäletama,NaN,el,mees,mehest,Pool sajandit tagasi pool aastat abielus olnud proua ei mäleta enam mehest suurt midagi .,NaN,NaN,NaN,no,NaN
316,6729182,10825176,11,tundma,NaN,el,arst,arstist,""" Kalinina möönis siiski , et sünnitusosakond tunneb puudust ühest arstist .",NaN,alive,NaN,no,NaN
350,9200938,14788385,11,mõistma,NaN,all,isa,isale,Rekordilise narkolasti tänavu märtsis Lõuna-Ameerikast kullerina Eestisse toimetanud viieaastase lapse isale Jaanus Ehalale ( 24 ) mõistis kohus kokaiini salakaubaveo eest nelja ja poole aasta pikkuse vanglakaristuse .,NaN,alive,NaN,no,NaN
358,13043,22652,2,vaatama,järele,all,mees,mehele,Millisele mehele vaatad tänaval järele ?,NaN,NaN,NaN,no,NaN
1013,1568110,2495358,1,valutama,NaN,ad,mees,Mehel,Mehel valutab kurk .,NaN,NaN,NaN,no,NaN
1176,677772,1077525,9,paluma,NaN,ad,arst,arstil,"Kui tahan apteegist odavamalt soodusravimeid saada , palun arstil kõik ravimid eraldi retseptidele kirjutada- nii annab apteek neist igaühe soodustusega .",NaN,alive,NaN,no,NaN
1382,13193356,21106165,4,moodustama,NaN,el,hambaarst,hambaarstist,Kiiruga moodustasime neljast hambaarstist aktsiaseltsi ja käisime idee välja .,NaN,alive,NaN,no,NaN
1582,7138440,11480445,8,saama,läbi,ad,mees,mehel,""" Kuule , meil saab siin ühel mehel katseaeg läbi .",NaN,NaN,NaN,no,NaN
1623,16749703,25774342,7,lõppema,NaN,ad,arst,arstidel,"Või on see selleks , et arstidel mitte kirjatöö otsa ei lõpeks , sest on ju nüüd vaja lisaks olemaolevate haigete lugude kirjutamisele ka vaja kirjutada lisakokkuvõtteid lugematutele lahkunutele ( mõtlen teise arsti juurde minijatele )",NaN,alive,NaN,no,NaN


In [106]:
live_idx4 = (no["lemma"]=="mina") | (no["lemma"]=="sina") |  (no["lemma"]=="tema") | (no["lemma"]=="meie") |  (no["lemma"]=="teie") | (no["lemma"]=="nemad")

no[live_idx4]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
88,8648447,13875590,8,kurtma,NaN,ad,mina,meil,"See peaks koolihariduse juurde kuuluma , ent meil kurdab staazhikas kriitikki et paradoksideklassik Becketti paradoksidetihedaid mõttekäike on öisel ajal raske jälgida ja kõik ei pruugi kohale jõuda , nagu ühest vaadeldava ööteatri järelkajast lugeda oli .",NaN,NaN,NaN,no,NaN
138,15612047,24363779,9,mainima,NaN,ad,mina,mul,"Midagi sellist , mida sa mainisid ei olnud mul küll mõttes ja arvan , et ka Juss ei mõelnud seda nii .",NaN,NaN,NaN,no,NaN
365,156042,259912,14,valutama,NaN,ad,tema,neil,"MM : On inimtüüpe , kes hakkavad hambaarstiks õppima selle pärast , et neil hammas valutab .",NaN,NaN,NaN,no,NaN
416,6811115,10958035,3,tegema,kaasa,ad,mina,meil,"Samas pole meil üheski mängus kaasa teinud kaks-kolm kogenut liidrit , kes vaevlevad vigastuse küüsis , "" lausus Klemann .",NaN,NaN,NaN,no,NaN
538,19821620,29300214,12,nihutama,NaN,ad,mina,mul,nica: seebitaja- kas sa pliis nihutaksid oma keelt pisut paremale mul oma kelel kitsas,NaN,NaN,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9393,16526860,25485600,4,ühinema,NaN,ad,tema,nendel,"See ettepanek võimaldaks nendel ühinenud volikogudel töötada sama kaua , kui töötavad teised volikogud , ja sellega me väldiksime Eestis iga omavalitsuste ühinemisega tekkivaid valimiskampaaniaid ning see võimaldaks säilitada meie riigis stabiilsust ja rahu .",NaN,NaN,NaN,no,NaN
9481,3728267,6004064,1,astuma,NaN,ad,mina,Meil,"Meil astuvad sportlased , näitlejad ja meediaankrud parteisse siis , kui neile pakutakse seal karjäärivõimalust .",NaN,NaN,NaN,no,NaN
9525,1661602,2645026,1,pruukima,NaN,all,mina,Mulle,"Mulle ei pruugi tingimused sobida . """,NaN,NaN,NaN,no,NaN
9744,7507118,12053595,16,tulema,kohale,ad,tema,tal,"See , et tal on võimalus tellimus vormistada nende hindadega ühe kuu jooksul , tuleb tal juba oma mõttetööga kohale , seletan irooniakillukesega hääles .",NaN,NaN,NaN,no,NaN


In [51]:
# -line lõpilune
no[no["lemma"].str.endswith("line")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
17,12537089,20066544,7,esitlema,NaN,all,piduline,pidulistele,Sajandi meeldejäävaimaist hetkedest tehtud filmi esitleb pidulistele Steven Spielberg .,NaN,alive,NaN,no,NaN
66,3815071,6145085,12,võtma,välja,ad,osaline,osalistel,"Euroopa Liidu tippkohtumine Amsterdamis võttis kolmapäeva hommikutundidel lõppedes võhma välja kõigil osalistel , Kok aga oli see mees , kes kõiki erimeelsusi ükshaaval 14 ja enama liidriga lahendada püüdis .",NaN,NaN,NaN,no,NaN
148,11171935,17895787,7,tasuma,NaN,ad,eluseltsiline,eluseltsilisel,"Selliseks kriitiliseks ajaks , mil tema eluseltsilisel tasuks eriti valvas olla , peavad seksuoloogid aastaid 35-36.",NaN,alive,NaN,no,NaN
331,4638160,7453047,1,esinema,NaN,ad,Suurejooneline,Suurejoonelisel,"Suurejoonelisel muusikaauhindade üleandmise sõul esinevad Coldplay , Kaiser Chiefs , KT Tunstall , Kanye West , Kelly Clarkson , James Blunt ja vanameister Paul Weller .",NaN,NaN,NaN,no,NaN
354,9385819,15072006,24,lisama,NaN,ad,viiesajaline,viiesajalistel,"Aga me oleme meeskonnas ( Marlboro Team Kanemoto Honda ) saavutanud hea tasakaalu , ” ütles 29 esikohale 250 cm3 klassis esimese ka viiesajalistel lisanud Biaggi pärast võistlust Reuterile .",NaN,NaN,NaN,no,NaN
515,11709343,18739731,10,jääma,üle,ad,lugemishuviline,lugemishuvilisel,"RAAMATUKOGUS : Et raamat kallineb järjest , siis jääb lugemishuvilisel üle oma kultuurijanu raamatukogus vaigistamas käia .",NaN,NaN,NaN,no,NaN
667,8132565,13052381,5,võtma,ära,abl,piduline,pidulistelt,"Politseinikud võtsid koostöös turvameestega pidulistelt ära 32 kahtlast tabletti , kaheksa kotikest rohelise puruga , neli kotikest valge pulbriga , kaks taimse puruga topitud piipu ja kaks kahtlase ainega täidetud sigaretti ning ühe annuse paberisse volditud taimepuru .",NaN,alive,NaN,no,NaN
702,16404088,25346220,15,vaatama,vastu,all,moehuviline,moehuvilisele,"Ly Ranne ( 35 ) oli üks kümnest Tallinna Moemaja modellist , kelle nägu moehuvilisele 80-ndate Silueti kaanelt vastu vaatas .",NaN,NaN,NaN,no,NaN
1275,6342684,10191872,1,tasuma,NaN,ad,joogahuviline,Joogahuvilistel,"Joogahuvilistel tasub külastada lehekülge , kus kirjas joogatreeningud üle Eesti .",NaN,NaN,NaN,no,NaN
1350,15882268,24712861,1,tegema,NaN,ad,Tavaline,Tavalisel,"Tavalisel teebki pildi vilkuvaks see , et vahepeal kaob hetkeks pilt ära st. kui elektronkiir on käinud üle luminofooride , siis see kiirgab korraks valgust , kuid kohe kaob see ära -sellest see virvendus .",NaN,NaN,NaN,no,It was not classified as adverbial of place because 'Tavalisel' does not indicate a location but rather refers to a general condition or manner.


In [79]:
# võimalikud nimed
name_idx = (no["head_loc"]>1) & (no["form"].str[0].str.isupper())

names = no[name_idx]

names


,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
6,10299850,16529878,9,laskma,NaN,ad,KMÜ,KMÜl,Miks te seda kohe ei öelnud ja lasite KMÜl veel koalitsioonileppe projekti ette valmistada ?,NaN,NaN,NaN,no,NaN
51,11394702,18259707,3,valmistama,NaN,all,Bondarenko,Bondarenkole,"Lisarõõmu valmistas Bondarenkole tõsiasi , et punkt saadi tänavu esmakordselt looduslikul murul mängides .",NaN,NaN,PER,no,"The phrase 'Bondarenkole' refers to the recipient of the action (dative case) rather than indicating a place, so it is not an adverbial of place."
52,127508,213568,11,lubama,NaN,ad,Kalev,Kalevil,"See lõi teadagi lõunamaalaste mänguplaani põhjalikult sassi , mis lubas Kalevil otsustava vahe skoori sisse raiuda .",NaN,alive,PER,no,NaN
53,1063467,1690574,4,mõistma,NaN,all,Maarika,Maarikale,"Hea õpetaja tähendab Maarikale mõistvat inimest , sellist , nagu oli õpetaja Laur .",NaN,NaN,LOC,no,NaN
65,6675804,10741057,5,külastama,NaN,ad,rai,Rail,Teisipäeval külastasid Valgat projekti Rail Baltica Euroopa Komisjoni poolne koordinaator Pavel Telička ja Tšehhi investeerimisfirma OKD Doprava esindajad .,NaN,alive,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9929,13550618,21669801,26,kirjutama,NaN,ad,Have,Havel,"Kundera , Klima , ja siis veel see vägev absurdist , noh , ütle nüüd selle mehe nimi , kirjutab natuke meie Valtoni moodi - Havel , muidugi Václav Havel ! """,NaN,NaN,NaN,no,NaN
9939,12903845,20640987,3,käskima,NaN,ad,Argo,Argol,Anu käsib Argol kõik riided seljast võtta .,NaN,NaN,PER,no,NaN
9962,5340112,8566570,2,minema,NaN,ad,Helen,Helenil,""" Helenil läksid silmad hirmust suureks , kui korravalvur lähenes .",NaN,NaN,NaN,no,NaN
9988,10255421,16456773,3,lõppema,NaN,ad,Rahmie,Rahmiel,Nii lõppes Rahmiel Blumbergi seitsmes ja teadaolevalt viimane isiklik retk Siberisse lapsi tooma .,NaN,NaN,NaN,no,NaN


In [87]:
# tegija
ija_idx = no["lemma"].str.endswith("ija")

no[ija_idx]



,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
3,750912,1196732,14,virutama,NaN,all,reisija,reisijale,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",NaN,alive,NaN,no,NaN
104,9159616,14725035,4,kippuma,NaN,ad,tantsija,tantsijatel,"“ Ka kipuvad tantsijatel aeg-ajalt jalad sõlme minema , ” märkis Jansu .",NaN,alive,NaN,no,NaN
573,16727992,25737499,13,tasuma,NaN,ad,kinkija,kinkijal,"Et roos ka tõepoolest kaks või enam nädalat vastu peaks , tasuks kinkijal eelistada kodumaist värsket lille .",NaN,NaN,NaN,no,NaN
645,8747132,14041300,6,kubisema,NaN,el,tegija,tegijatest,Kohalike valimiste nimekirjad kubisevad vanadest tegijatest,NaN,NaN,NaN,no,NaN
781,9115631,14656103,13,tasuma,NaN,ad,järeltulija,järeltulijal,"Arvan veendunult , et Päts oli tõeline riigimees , kellest igal tema järeltulijal tasuks eeskuju võtta just selles , kuidas üks elu on jäägitult elatud Eestile .",NaN,alive,NaN,no,NaN
958,3762700,6060401,3,võtma,vastu,abl,hankija,hankijalt,Siseministeerium võtab hankijalt vastu Schengen Facility programmi abil piirivalvele ning maksu- ja tolliametile muretsetud kaks mobiilset läbivalgustusseadet .,NaN,NaN,NaN,no,NaN
1064,3717315,5986727,12,kehtima,NaN,all,kojuviija,kojuviijatele,Ettevõtlusameti juhataja Kairi Teniste sõnul kehtib alkoholimüügi keeld siiski ka toidu kojuviijatele .,NaN,NaN,NaN,no,NaN
1237,4363150,7018921,15,pooldama,NaN,el,valija,valijatest,""" President Lukashenko konstitutsiooni ei tohi tunnustada isegi siis , kui seda pooldas 101% valijatest , "" lausus Tihhinja .",NaN,NaN,NaN,no,NaN
1405,7321435,11752441,33,seletama,NaN,all,olija,olijatele,"Keelejuht PH on nii neid sõnu kui järgmise lause sõnu ( ei sööda , ei süüdä ) hääldanud kaks korda , sest ta tegi pärast lausete lugemist pausi ja seletas lindistuse juures olijatele , mida sõnad tähendavad , öeldes sõnad uuesti .",NaN,alive,NaN,no,NaN
2041,18495194,28008898,2,soovitama,NaN,ad,arupärija,arupärijatel,"Soovitan arupärijatel - kuigi ma saan aru , et nad peavad olema valitsuse suhtes kriitilised - hinnata ka seda tõsiasja , mis avaldati eelmisel nädalal .",NaN,NaN,NaN,no,"The phrase 'arupärijatel' refers to the individuals being addressed, not a location, so it is not adverbial of place."


In [100]:
# tegija2
ija2_idx = no["lemma"].str.endswith("ja")

no[ija2_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
3,750912,1196732,14,virutama,NaN,all,reisija,reisijale,"Kaks aastat tagasi oli Eesti Päevalehe ajakirjanik tunnistajaks , kuidas kontrollija piletita sõitnud reisijale jalaga vastu tagumikku virutas ja vihaselt sõimles .",NaN,alive,NaN,no,NaN
10,13016963,20826469,3,karjuma,NaN,all,kiskuja,kiskujale,"« Karjusin kiskujale : ei , ei , lükka auto püsti !",NaN,NaN,NaN,no,"The phrase 'kiskujale' refers to a person, not a place, so it is not an adverbial of place."
73,508489,800245,18,muutuma,NaN,all,kalastaja,kalastajatele,"Alates esmaspäevast keelas Pärnu maavanem Toomas Kivimägi jäälemineku ka Pärnu lahel , sest nõrk jää on muutunud kalastajatele eluohtlikuks .",NaN,alive,NaN,no,NaN
75,6645362,10692993,19,karjuma,NaN,all,turvaja,turvajale,""" Siin kamandan mina , teie istuge aga autosse , "" karjus heledapäine turvaülem vene keeles patriarhi Vene-poolsele turvajale , kes ilmselt ei soovinud nii tulise rutuga lennuväljalt lahkuda , kui kohalik turvaplaan ette nägi .",NaN,alive,NaN,no,NaN
104,9159616,14725035,4,kippuma,NaN,ad,tantsija,tantsijatel,"“ Ka kipuvad tantsijatel aeg-ajalt jalad sõlme minema , ” märkis Jansu .",NaN,alive,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9840,16415159,25362655,5,jätkama,NaN,el,alavõitja,alavõitjast,"12 Kuldliiga esimese etapi alavõitjast jätkavad heitlust miljoni dollari suuruse jackpot'i eest seitse – Wilson Kipketer , Allen Johnson , Bernard Barmasai , Erick Walder , Marion Jones , Svetlana Masterkova ja Gabriela Szabo .",NaN,NaN,NaN,no,NaN
9843,1731199,2755658,8,maksma,välja,all,sõlmija,sõlmijale,Hanspanga Kindlustus maksab ligi 40 000 kogumiskindlustuse lepingu sõlmijale tänavu garanteeritud ja lisaintressidena välja ligi 10 miljonit krooni .,NaN,NaN,NaN,no,NaN
9868,696768,1108624,4,meenuma,NaN,all,bensiinitarbija,bensiinitarbijale,"Samas meenub mõnele bensiinitarbijale kindlasti , et mullu , kui toornafta hind kolinal langes , ei teinud bensiini hind seda mitte .",NaN,NaN,NaN,no,NaN
9877,7094951,11409288,3,tulema,järele,all,näitleja,näitlejale,Pärast oli näitlejale naine kodust järele tulnud ja mehe puhkama viinud .,NaN,alive,NaN,no,NaN


## no hulgas timex

In [65]:
day_idx = (no["form"].str.lower().str.contains("päev"))

no.loc[day_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
55,7597772,12192312,5,tüdinema,NaN,el,päev,päevast,“ Ma olin tüdinenud päevast päeva osa võtmast igasugu juhatuste ja nõukogude koosolekutest Rootsis .,NaN,NaN,NaN,no,NaN
77,113589,190849,25,täituma,NaN,ad,päev,päevadel,"Uusasundused , mille ehitus Peresi ajal peatati täielikult - isegi valminud majadesse ei lubatud uusi elanikke sisse kolida - , täitusid Netanyahu valitsemisaja esimestel päevadel .",NaN,NaN,NaN,no,NaN
82,136287,227781,6,algama,NaN,ad,aastapäev,aastapäeval,"Hämmastaval kombel algas just küüditamise aastapäeval Euroopas , mõne tuhande kilomeetri kaugusel veelgi vägevam küüditamine .",NaN,time,NaN,no,NaN
85,10519936,16865184,4,kaitsma,NaN,ad,teisipäev,teisipäeval,Jaak Allik kaitseski teisipäeval Mart Opmanni ja Tiiu Aro käitumist .,NaN,time,NaN,no,NaN
141,2986801,4790884,17,arvestama,NaN,ad,vanaduspäev,vanaduspäevil,"Kuidas see meelekohalt hall ja elukogenud mees seda täisväärtuslikku elu elada saab , kui riik talle vanaduspäevil vaid sandikopikaid arvestab .",NaN,NaN,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,5171180,8296564,2,kaitsma,NaN,ad,tennisepäev,tennisepäeval,"IBMi tennisepäeval kaitses Luzgin ka tugevama soo õigusi , alistades Kanepi 6 : 1 , 6 : 1.",NaN,NaN,NaN,no,NaN
9851,2588613,4154554,10,kohtuma,NaN,ad,ametipäev,ametipäeval,Taas kohtusid kaevurid uue juhiga esimest korda just nimelt ametipäeval .,NaN,NaN,NaN,no,NaN
9869,15607472,24358419,5,tulema,kohale,ad,laadapäev,laadapäeval,Aga eks sinna tuleb laadapäeval ka teisi müüjaid kohale .,NaN,time,NaN,no,NaN
9930,11544316,18486590,6,valima,välja,ad,päev,päeval,"On ilmselge , et päikeselisel päeval valite testi vastusevariantidest välja optimistlikumad , krõbeda pakasega ” Kas oled seksikas ?",NaN,NaN,NaN,no,"The phrase 'päeval' indicates a time frame, not a location, so it is not adverbial of place."


In [61]:
idx1 = (no["form"].str.lower().str.contains("jaanuar")) | (no["form"].str.lower().str.contains("veebruar")) | (no["form"].str.lower().str.contains("märts"))
idx2 = (no["form"].str.lower().str.contains("aprill")) | (no["form"].str.lower().str.contains("mai")) | (no["form"].str.lower().str.contains("juuni"))
idx3 = (no["form"].str.lower().str.contains("juuli")) | (no["form"].str.lower().str.contains("august")) | (no["form"].str.lower().str.contains("septemb"))
idx4 = (no["form"].str.lower().str.contains("oktoob")) | (no["form"].str.lower().str.contains("novemb")) | (no["form"].str.lower().str.contains("detsemb"))

no.loc[idx1 | idx2 | idx3 | idx4]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
120,14678401,23140123,2,jääma,ära,ad,oktoober,oktoobril,16. oktoobril jääb sotsiaalantropoloogia loeng ära ; selle asemel on 23. oktoobril lisaks loengule seminar .,NaN,NaN,NaN,no,It was not classified as adverbial of place ('no') because 'oktoobril' refers to a time frame rather than a location.
235,13910306,22183181,2,nõudma,NaN,ad,aprill,aprillil,"13. aprillil nõudsid üheksa USA kongresmeni oma avalduses Venemaalt selget ülestunnistust , et Nõukogude Liit okupeeris ja annekteeris kolm iseseisvat Balti riiki .",NaN,NaN,NaN,no,NaN
327,13656111,21823521,16,pühkima,NaN,ad,detsember,detsembril,"Kui Aus jääb Eestimaale paariks nädalaks , siis Kirsipuu pühib siinse tolmu jalgelt juba 28 . detsembril .",NaN,time,NaN,no,NaN
486,9417728,15122096,2,mainima,NaN,ad,märtsiõhtu,märtsiõhtul,"Ühel märtsiõhtul mainis Toomas Savi Mart Siimanni telelauas , et milleks võõrapärane spiiker , ta eelistaks olla eestipäraselt Riigikogu esimees .",NaN,NaN,NaN,no,NaN
622,17984026,27365736,14,katkema,NaN,ad,oktoober,oktoobril,"Sügise omapäraks tuleb pidada sedagi , et taimede põhiline elutegevus katkes juba 4. oktoobril , paljude aastate keskmisest 17 päeva varem .",NaN,NaN,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9459,10724656,17184907,2,hoiatama,NaN,ad,aprill,aprillil,"4 . aprillil hoiatas Mihhail Gorbatšov Arnold Rüütlit : kui ENSV Ülemnõukogu ei tühista oma otsust riikliku staatuse kohta , siis rakendab NSVL Eesti suhtes samasuguseid sanktsioone nagu Leedus ( majandusblokaad ja tankide viimine Vilniusse ) ;",NaN,NaN,NaN,no,NaN
9473,13369516,21387708,6,ühinema,NaN,ad,aprill,aprillil,Parempoolsed ja ETRE ühinevad 5 . aprillil Tartus .,NaN,NaN,NaN,no,NaN
9771,10936506,17516193,2,määrama,NaN,ad,juuli,juulil,12. juulil määras projekti juhtkomitee sihtpiirkonnaks ka Tartumaa .,NaN,time,NaN,no,NaN
9830,9672751,15522240,3,käivituma,NaN,el,oktoobrikuu,oktoobrikuust,1996. aasta oktoobrikuust käivitus koostöös Soome Nägemisvaegurite Keskliidu ja Soome Rootsikeelsete Nägemispuudeliste Liiduga suur Phare Lieni projekt .,NaN,time,NaN,no,It is not classified as adverbial of place because 'oktoobrikuust' refers to a point in time rather than a location.


## lemma on -mine lõpuline

In [95]:
mine_idx = (no["lemma"].str.endswith("mine"))

no.loc[mine_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
7,7950871,12740282,35,suhtuma,NaN,adit,idealiseerimine,idealiseerimisse,"Ma usun , et lugejad siiski tajusid , vastupidi , ideoloogiavastast subversiooni , õõnestust või vähemalt ambivalentsust , sest tegelikult oli Hansen-Tammsaare antifašist ning suhtus oma teostes kriitiliselt ka talupoegluse ning "" juurte "" idealiseerimisse .",NaN,NaN,NaN,no,NaN
9,95952,162188,10,tulema,kokku,ad,lõpetamine,lõpetamisel,"“ Ühelt poolt on muidugi hea , et keskkooli lõpetamisel ja ülikooli sisseastumisel ei tule kokku ligi kümmet eksamit teha , teisalt võetakse mul riigieksamitega teine võimalus .",NaN,event,NaN,no,NaN
20,18514356,28036088,5,kuulama,NaN,ad,menetlemine,menetlemisel,Praegu me oleme eelarve menetlemisel kannatlikult kuulanud opositsiooni kriitikat .,NaN,NaN,NaN,no,NaN
21,4120887,6633233,8,teavitama,NaN,ad,eemaldamine,eemaldamisel,Samuti teavitavad Tallinn ja Helsingi vastastikku grafiti eemaldamisel ja pindade kaitsmisel kasutatavatest uutest tehnoloogiatest jms.,NaN,NaN,NaN,no,NaN
40,16682497,25670387,11,selgitama,NaN,ad,menetlemine,menetlemisel,"Aga mis puudutab Siseministeeriumi , siis nagu ma juba eelarve menetlemisel selgitasin , seisab ees selle ja ka ametite ümberkorraldamine .",NaN,NaN,NaN,no,"The phrase 'menetlemisel' refers to the process of handling or discussing the budget and is not describing a location, so it is not an adverbial of place."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9949,6802132,10944118,2,koostama,NaN,ad,tekkimine,tekkimisel,"Võlgade tekkimisel koostab Ehituse ABC väikeettevõtjale maksegraafiku , kasutades vajadusel ka inkassofirma teenuseid .",NaN,NaN,NaN,no,NaN
9955,13581967,21717573,2,kuulutama,NaN,ad,taasiseseisvumine,taasiseseisvumisel,"Kuigi taasiseseisvumisel kuulutasid mõned prominendid põllumajanduse perspektiivituks alaks , pole see põhjendatud .",NaN,NaN,NaN,no,NaN
9968,4218116,6788151,3,lisanduma,NaN,ad,eestvedamine,eestvedamisel,"Liis Valgu eestvedamisel lisandus tänavu vanalinna piirkonda neli uut lillepeenart , kümme lilletorni ja 60 uut lillevaasi .",NaN,NaN,NaN,no,NaN
9981,10835987,17354786,9,vedama,alt,ad,allalaskumine,allalaskumisel,Juhtimissüsteem ja langevari vedasid alt Vene Sojuzi kapsli allalaskumisel .,NaN,NaN,NaN,no,"It is not classified as an adverbial of place because 'allalaskumisel' ('during the descent') refers to time or process, not location."


### lemma on "kord"

In [111]:
kord_idx = (no["lemma"].str.endswith("kord"))

no.loc[kord_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
0,7108634,11432411,7,tasuma,NaN,ad,kord,korral,"“ Ei tea , kas esimesel korral tasub lõpuni minna , ” kahtlesin .",NaN,NaN,NaN,no,"The phrase 'korral' refers to 'occasion' and does not indicate a location, hence it is not adverbial of place."
38,2101376,3353927,11,juhinduma,NaN,el,sisekord,sisekordadest,"Nendes küsimustes langetavad otsuse meie piirkonnad , kes juhinduvad meie sisekordadest ja eetikakoodeksist .",NaN,NaN,NaN,no,NaN
960,6156517,9882170,6,diagnoosima,NaN,ad,kord,korral,"Grippi diagnoosisid arstid vaid paaril korral ja sedagi kliinilise pildi , mitte analüüside põhjal .",NaN,NaN,NaN,no,"The phrase 'korral' refers to an instance or occasion rather than a physical location, so it is not adverbial of place."
1032,2834839,4546517,3,kärpima,NaN,ad,kord,kordadel,Ka varasematel kordadel on energiaturuinspektsioon Eesti Energia hinnatõstmisplaane kärpinud ja Atoneni hinnangul püüavad Eesti Energia juhid inspektsiooni üle kavaldada .,NaN,NaN,NaN,no,NaN
1335,15447761,24126591,2,jääma,sisse,ad,kord,korral,"Neljandal korral jäime sisse neljaks nädalaks , esinesin siis koos Maian Kärmase , Helen Tartese , Pavel",NaN,NaN,NaN,no,NaN
2131,4422093,7111377,16,vaidlema,vastu,all,erastamiskord,erastamiskorrale,"Eramu- , suvila- ja aianduskruntide sooduserastamise kõige vihasemadki vastased pole avalikult vastu vaielnud praegu kehtivale erastamiskorrale , vaid seda koguni propageerinud .",NaN,NaN,NaN,no,NaN
2473,187418,311149,2,osutuma,NaN,ad,kord,korral,Vastasel korral osutub tehnoloogia nõrgumine naiivseks ja liigoptimistlikuks ettekujutuseks .,NaN,NaN,NaN,no,NaN
2590,14368477,22792679,29,soovitama,NaN,all,kord,korrale,"b ) Kui korraldusasutusele on soovitatud punktis a nimetatud meetmeid , teatab ta nendest ja annab oma selgitused komisjonile , kes vajaduse korral soovitab vastavalt artiklis 18 ettenähtud korrale nende liikide ekspordipiiranguid .",NaN,NaN,NaN,no,The phrase 'korrale' was classified as not adverbial of place ('no') because it does not denote a physical location but rather refers to a procedural order.
2692,10121924,16239521,11,minema,järele,ad,kord,korral,Korralik piibel oli tollal suur rariteet ja muidugi läksin järgmisel korral sellele järele .,NaN,NaN,NaN,no,NaN
3392,1359763,2161560,18,nõudma,NaN,ad,kord,korral,"Sharif mõisteti süüdi terrorismis ja lennukikaaperdamises ja talle määrati kahekordne eluaegne vanglakaristus , kuigi süü-distaja nõudis kahel korral peaministriks valitud ja mõlemal korral kukutatud Sharifile lennukikaaperdamise eest surmanuhtlust .",NaN,NaN,NaN,no,NaN


In [117]:
aeg_idx = (no["lemma"].str.lower().str.contains("aeg"))

no.loc[aeg_idx]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
4,4733683,7604499,14,lülitama,NaN,ad,poolaeg,poolajal,""" Seda ma paraku ei tea , sest lülitasin teleka sisse alles teisel poolajal . """,NaN,time,NaN,no,NaN
39,4033468,6491658,6,kaasama,NaN,ad,hooaeg,hooajal,"Kristina seltskond on eriti sel hooajal nutikalt kaasanud inimesi , kes on toonud endaga mõistmise , et vaid nüüdisteaduse tasemel saab kroonida võitjaeeldustega isiku tõeliseks võitjaks .",NaN,time,NaN,no,NaN
94,14048138,22366678,6,süüdistama,NaN,ad,valitsusaeg,valitsusajal,USA süüdistas Nijazovit tema autokraatlikul valitsusajal inimõiguste rikkumistes .,NaN,time,NaN,no,NaN
342,10434720,16734305,5,hõivama,NaN,ad,hooaeg,hooajal,"Peatreeneri koha hõivab sel hooajal Malmö klubi juhendanud Hannu Jortikka , abitreenereiks on nimekad mängumehed Hannu Virta ja Kari Jalonen .",NaN,time,NaN,no,NaN
347,2761964,4429484,2,luhtuma,NaN,ad,avapoolaeg,Avapoolajal,""" Avapoolajal luhtus kolm üks-üks võimalust , tehti rumalaid viskeid , selge , et praaki tuleb vähendada , "" lisas Musting .",NaN,NaN,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9317,739390,1178230,14,võistlema,NaN,ad,aeg,ajal,"Erakonnad konkureerivad eelkõige omavahel , kuid valijate tähelepanu võitmiseks meedias võistlevad nad samal ajal ka kommertskampaaniatega .",NaN,NaN,NaN,no,NaN
9529,1705658,2714787,5,määrama,NaN,ad,poolaeg,poolajal,"Kui Levnikov oli esimesel poolajal Maroko kasuks vabalöögi määranud , läksid kaks nimekat meest , teist MMi kaptenipaela kandev Dunga ja Brasiilia nelja aasta taguses triumfis tähtsat tööd teinud Bebeto isekeskis põhjalikult tülli .",NaN,time,NaN,no,NaN
9568,14061961,22389708,4,suruma,NaN,ad,kellaaeg,kellaajal,"Kui täpselt ühel kellaajal suruvad end kabinetti nii draamainimesed kui ka haldusmehed ja kõigil on sada muret tagataskus , siis toimub juhi kabinetis midagi mesilaste moosisaiatantsu taolist .",NaN,NaN,NaN,no,NaN
9669,21162,36289,4,kaduma,ära,ad,aeg,ajal,"“ Kui omal ajal riigiraadios diktori amet ära kadus , olin paar aastat saatejuht .",NaN,NaN,NaN,no,NaN


## mis on muu

In [152]:


muu1 = no[~num_idx & ~lane_idx & ~day_idx & ~name_idx & ~idx1 & ~idx2 & ~idx3 & ~idx4 & ~live_idx1 & ~live_idx2 & ~live_idx3 & ~live_idx4 & ~ija_idx & ~ija2_idx & ~mine_idx & ~kord_idx & ~aeg_idx ]
#muu1


In [168]:
juht = (muu1["lemma"].str.endswith("juht"))
aasta = (muu1["lemma"].str.contains("aasta"))
ohtu = (muu1["lemma"].str.contains("õhtu"))

muu2 = muu1[~juht & ~aasta & ~ohtu]

In [154]:
aggreg2 = muu2.groupby(["lemma"], dropna=False).agg(
        lemmas = ("lemma", lambda x :len(x))
).sort_values('lemmas', ascending=False)
aggreg2

,lemmas
lemma,
andmed,23
hetk,22
kes,22
kinnitus,21
alune,20
...,...
hoiatus,1
hoiak,1
hobune,1


In [155]:
aggreg2.head(30)

,lemmas
lemma,
andmed,23
hetk,22
kes,22
kinnitus,21
alune,20
teade,19
mis,17
hinnang,16
põhjus,16


In [156]:
muu2

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
1,2601827,4175718,12,pakkuma,NaN,ad,juhatus,juhatusel,"Eesti Filharmoonia Kammerkoor ja Tallinna Kammerorkester pakuvad täna õhtul Tõnu Kaljuste juhatusel Tallinna Metodisti kirikus toimuval kontserdil valiku teostest , mida koor ja orkester esitavad reedel algaval kolmenädalasel USA ja Kanada turneel .",NaN,NaN,NaN,no,"The phrase 'juhatusel' refers to 'under the direction' and is describing leadership or guidance, not location, so it is not adverbial of place."
2,1677278,2670999,3,seletama,NaN,all,patsient,patsiendile,"Kullamaa seletab patsiendile alati , et iga protees on organismile võõrkeha ja ümber selle tekitab organism poole aasta jooksul sidekoelise kapsli .",NaN,NaN,NaN,no,NaN
5,14922621,23437007,3,külastama,NaN,ad,nahkkuu,nahkkuul,"Pealegi külastas nahkkuul kahel korral mulkide väravaraamistikku , ohtlikke olukordi jätkus veelgi .",NaN,NaN,NaN,no,NaN
8,15337147,23952644,11,tutvustama,NaN,ad,hulk,hulgal,Ulmekirjandus ja -filmid on meile viimastel aastakümnetel tutvustanud arvutul hulgal mõtlemisvõimelisi masinaid .,NaN,NaN,NaN,no,NaN
13,10517633,16861298,1,mõjuma,NaN,all,aatom,Aatomitele,"Aatomitele , mis käituvad pisimagnetitena , mõjub raskusjõust suurem jõud ning siirup ei saa üle magnetkausi servade välja voolata .",NaN,NaN,NaN,no,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9978,1510157,2403680,10,tegema,kaasa,ad,mood,moel,"Kui Sjoma loeb üle kõik sõbrad , kes mingil moel kaasa teinud/teevad , tuleb nimesid kahekümne ümber .",NaN,NaN,NaN,no,NaN
9982,10423906,16721610,13,suhtuma,NaN,adit,käepigistus,käepigistusse,"Iseasi , et teenetemärkide all lookas filmid suhtuvad ilmse ükskõiksusega Alliku sooja käepigistusse ja Seaküla Simsoni Kappava Hobuse võiduskulptuuri .",NaN,NaN,NaN,no,NaN
9983,4769521,7659715,27,toetama,NaN,ad,kaasabi,kaasabil,"Läbirääkimiste eelmises voorus tööandjate esindajad oma konkreetseid numbreid välja ei käinud , küll aga väljendasid nad põhimõttelist valmisolekut alampalka tõsta , kui riik Euroopa Liidu tõukefondide kaasabil toetaks suuremas mahus ettevõtete investeeringuid teh-noloogia uuendamisse ja aitaks nii kaasa tööviljakuse kasvule .",NaN,NaN,NaN,no,NaN
9987,8960373,14404705,14,tooma,NaN,ad,kviitung,kviitungil,"Tellite ja sööte , kui palju soovite , lõpus toob kelner teile arve kviitungil , millele te kirjutate oma toanumbri ja allkirja , arve dublikaat jääb teile .",NaN,NaN,NaN,no,NaN


In [149]:
muu2[muu2["lemma"].str.contains("Artur")]

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
7713,8583913,13759298,1,minema,NaN,ad,Artur,Arturil,"Arturil ja Krislinil läheb õppimine hästi , Richardile on see aga raskem .",NaN,NaN,PER,no,NaN


In [182]:
words = muu2["lemma"].unique()
words = sorted(words, key=len)

groups = {}

for w in words:
    found = False
    for base in groups:
        #print(w, base)
        if len(base) >=4:
            if base in w or w in base:
                groups[base].append(w)
                found = True
                break
    if not found:
        if len(w)>=4:
            groups[w] = [w]



In [185]:
groups["hääl"]

['hääl',
 'hääldus',
 'hääletus',
 'naisehääl',
 'hääletustund',
 'näitlejahääl',
 'suitsetajahääl']

In [181]:
mapping = {w: base for base, ws in groups.items() for w in ws}

muu3 = pd.DataFrame()
muu3["aggregated"] = muu2["lemma"].map(mapping)

muu3 = muu3.groupby("aggregated").size().reset_index(name="count")
muu3 = muu3.sort_values("count", ascending=False)
muu3

,aggregated,count
458,hetk,37
177,andme,29
943,nädal,27
718,kuju,25
1107,seis,24
...,...,...
27,Hafsteinsson,1
12,Bosnia,1
13,Bowe,1
14,Budõlini,1


## Kui palju saaks üldse ette anda gpt-le märgendamiseks

In [40]:
df4 = pd.read_csv("../../data/n20_examples_large_v01.csv", encoding="utf-8", sep=",")

In [103]:
len(df4)

120600

In [42]:
aggreg2 = df4.groupby(["verb", "verb_compound", "morph_case"], dropna=False).agg(
        lemmas = ("lemma", lambda x :len(x))
).sort_values('lemmas', ascending=False)
aggreg2

lemmas
verb      verb_compound morph_case        
andma     NaN           el             500
                        ad             500
üritama   NaN           ad             500
avaldama  NaN           ad             500
aitama    NaN           ad             500
...                                    ...
potsatama NaN           ad              18
tõstma    välja         ad              17
mattuma   NaN           ad              17
külmutama NaN           ad              17
minema    järele        ad              16

[534 rows x 1 columns]